# Proofreading fly-through demo

Clean replacement for the exploratory `original/Neuroglancer/ExASPIM_Tiff_in_NG-*.ipynb`
notebooks, built on the [`proofreading`](proofreading/) package.

Run from the repo root in the `uv` environment (`uv run jupyter lab`).


## 0. Imports

`ac_ngl` is the lab's helper module (not on PyPI) -- point `sys.path` at it.


In [ ]:
import sys
sys.path.append('/home/wanqing.yu/AC_Project/ac_visualization/')  # ac_ngl

import neuroglancer
import navis
import proofreading as pf

## 1. (optional) Build precomputed skeletons

Only needed once per dataset. Uses `ac_ngl`'s generators directly.


In [ ]:
# from ac_ngl import generate_ngl_segmentation_empty, generate_ngl_skeletons
# import os
#
# indir  = '/ACdata/Users/wanqing/exaSPIM/.../skeleton/skeletons_10'
# outdir = '/ACdata/Users/wanqing/Neuroglancer/ExASPIM/'
# os.makedirs(outdir, exist_ok=True)
# generate_ngl_segmentation_empty([288, 288, 21586], indir, outdir,
#                                 [512, 512, 64], [406, 406, 1997.72])
# generate_ngl_skeletons(indir, outdir, match_fname=False)

## 2. Viewer + layers


In [ ]:
viewer = pf.make_viewer(port=9998)

pf.load_image_layer(
    viewer, 'ExASPIM',
    '/ACdata/Users/wanqing/exaSPIM/precomputed/',
    shader_range=[15, 71],
)
pf.load_skeleton_layer(
    viewer, 'skeletons',
    '/ACdata/Users/wanqing/Neuroglancer/ExASPIM/skeletons/',
)
print(viewer)  # open this URL in the browser

## 3. Load a skeleton and build the camera path

`compute_path` returns `(positions, orientations)`: node coordinates in
`[x, y, z]` order plus the matching per-node camera quaternions
(tangent -> dotprops -> quaternion -> axis-flip, via `ac_ngl`).


In [ ]:
skel = pf.load_skeleton(
    '/ACdata/Users/wanqing/exaSPIM/.../skeletons_10/0002.swc',
    swap_xz=True,
)
positions, orientations = pf.compute_path(skel, k=15)
positions.shape, orientations.shape

## 4. Interactive fly-through

The buttons run on a background thread, so they stay responsive while the
camera is moving:

- **Play / Pause**, **Forward / Reverse** (autoplay direction)
- **Step ◁ / ▷** (one node), **scrubber** (jump anywhere), **sec/step** (speed)


In [ ]:
fly = pf.FlyThrough(viewer, positions, orientations, seconds_per_step=0.3)
controls = pf.FlyThroughControls(fly)  # renders the button panel

## 5. Show skeletons near the current position

Build a box around wherever the camera is now and reveal nearby segments.
Set `name_to_id_offset` to match your SWC-name -> segment-id convention
(this was the off-by-one the old notebooks kept flip-flopping on).


In [ ]:
skel_all = navis.read_swc('/ACdata/Users/wanqing/exaSPIM/.../skeletons_10/')

i = fly.index
center = positions[i]  # [x, y, z]
ids = pf.nearby_skeleton_ids(skel_all, center, size=(100, 100, 100),
                             name_to_id_offset=-1)
pf.context.show_segments(viewer, 'skeletons', ids)
ids

## 6. Shutdown


In [ ]:
fly.stop()
neuroglancer.stop()